# VILAGENT UI-TARS Target Resolution via pyngrok

This notebook exposes a JSON target-resolution service for VILAGENT. It also caches `UI-TARS-1.5-7B` in Google Drive at `My Drive/models/UI-TARS-1.5-7B` so the model is downloaded only once. Later sessions copy the Drive cache to fast Colab local disk before serving. Replace `load_uitars_model` and `resolve_with_uitars` with your final UI-TARS inference code; keep the request/response contract stable.

In [ ]:
!pip -q install -U fastapi uvicorn pyngrok nest_asyncio pillow pydantic huggingface_hub


In [ ]:
import os
from getpass import getpass

NGROK_AUTHTOKEN = os.getenv("NGROK_AUTHTOKEN") or getpass("NGROK_AUTHTOKEN: ")
UITARS_API_KEY = os.getenv("VILAGENT_UITARS_API_KEY") or getpass("VILAGENT_UITARS_API_KEY for VILAGENT: ")
MODEL_NAME = os.getenv("VILAGENT_UITARS_MODEL_NAME", "UI-TARS-1.5-7B")
HF_MODEL_ID = os.getenv("VILAGENT_UITARS_HF_MODEL_ID", "ByteDance-Seed/UI-TARS-1.5-7B")
PORT = int(os.getenv("VILAGENT_UITARS_PORT", "7860"))
os.environ["NGROK_AUTHTOKEN"] = NGROK_AUTHTOKEN
print({"model": MODEL_NAME, "hf_model_id": HF_MODEL_ID, "port": PORT})


In [ ]:
import shutil
from pathlib import Path
from google.colab import drive
from huggingface_hub import snapshot_download

drive.mount("/content/drive")

# Google Drive displays this as: My Drive/models/UI-TARS-1.5-7B
DRIVE_MODEL_DIR = Path("/content/drive/MyDrive/models/UI-TARS-1.5-7B")
LOCAL_MODEL_DIR = Path("/content/models/UI-TARS-1.5-7B")

def has_model_files(path: Path) -> bool:
    return path.exists() and any(path.iterdir())

def copy_drive_model_to_local() -> None:
    if LOCAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_MODEL_DIR)
    LOCAL_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_MODEL_DIR, LOCAL_MODEL_DIR)

if not has_model_files(DRIVE_MODEL_DIR):
    print(f"Drive cache missing. Downloading {HF_MODEL_ID} to {DRIVE_MODEL_DIR} ...")
    DRIVE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id=HF_MODEL_ID,
        local_dir=str(DRIVE_MODEL_DIR),
        local_dir_use_symlinks=False,
        resume_download=True,
    )
else:
    print(f"Drive cache found: {DRIVE_MODEL_DIR}")

print(f"Copying model from Drive cache to local runtime disk: {LOCAL_MODEL_DIR}")
copy_drive_model_to_local()
MODEL_PATH = str(LOCAL_MODEL_DIR)
print("MODEL_PATH=", MODEL_PATH)


In [ ]:
import base64, io, time
from typing import Any
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
from PIL import Image

app = FastAPI(title="VILAGENT UI-TARS Bridge")
started_at = time.time()

class ResolveRequest(BaseModel):
    model: str | None = None
    task: str = "target_resolution"
    description: str
    selector_hints: dict[str, Any] = Field(default_factory=dict)
    minimum_confidence: float = 0.5
    observation: dict[str, Any] = Field(default_factory=dict)
    image_base64: str | None = None
    screenshot_url: str | None = None

def check_auth(authorization: str | None):
    if not UITARS_API_KEY:
        return
    if authorization != f"Bearer {UITARS_API_KEY}":
        raise HTTPException(status_code=401, detail="invalid API key")

def decode_image(image_base64: str | None):
    if not image_base64:
        return None
    raw = image_base64.split(",", 1)[-1]
    return Image.open(io.BytesIO(base64.b64decode(raw))).convert("RGB")

def load_uitars_model(model_path: str):
    # TODO: Load UI-TARS from model_path once, for example with transformers/vLLM.
    # Keep this function isolated so Drive/local cache logic remains reusable.
    print(f"UI-TARS model cache ready at: {model_path}")
    return None

UITARS_MODEL = load_uitars_model(MODEL_PATH)

def resolve_with_uitars(request: ResolveRequest) -> dict[str, Any]:
    # TODO: Replace this fallback with real UI-TARS inference using UITARS_MODEL.
    # The fallback keeps the HTTP contract testable before model wiring.
    image = decode_image(request.image_base64)
    width, height = image.size if image is not None else (1920, 1080)
    x = width // 2
    y = height // 2
    confidence = max(float(request.minimum_confidence), 0.51)
    return {
        "found": True,
        "target": {
            "confidence": confidence,
            "bounds": {"x": x - 12, "y": y - 12, "width": 24, "height": 24},
            "selector": {"point": {"x": x, "y": y}, "source": "uitars-colab-template"},
            "label": request.description,
        },
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "model": MODEL_NAME,
        "hf_model_id": HF_MODEL_ID,
        "model_path": MODEL_PATH,
        "drive_cache": str(DRIVE_MODEL_DIR),
        "uptime_seconds": int(time.time() - started_at),
    }

@app.post("/resolve")
def resolve(request: ResolveRequest, authorization: str | None = Header(default=None)):
    check_auth(authorization)
    return resolve_with_uitars(request)


In [ ]:
import nest_asyncio, threading, uvicorn

nest_asyncio.apply()
thread = threading.Thread(target=lambda: uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info"), daemon=True)
thread.start()
print("UI-TARS bridge started on", PORT)


In [ ]:
from pyngrok import conf, ngrok

conf.get_default().auth_token = NGROK_AUTHTOKEN
tunnel = ngrok.connect(PORT, "http")
public_url = tunnel.public_url.rstrip("/")
print("Public base:", public_url)
print("\nCopy into .env:")
print("VILAGENT_UITARS_ENABLED=true")
print(f"VILAGENT_UITARS_MODEL_NAME={MODEL_NAME}")
print(f"VILAGENT_UITARS_PYNGROK_URL={public_url}")
print(f"VILAGENT_UITARS_API_KEY={UITARS_API_KEY}")
print("VILAGENT_UITARS_ENDPOINT_PATH=/resolve")
print("VILAGENT_UITARS_HEALTH_ENDPOINT_PATH=/health")


In [ ]:
import requests

headers = {"Authorization": f"Bearer {UITARS_API_KEY}"}
print(requests.get(f"{public_url}/health", headers=headers, timeout=30).json())
payload = {"description": "center test", "minimum_confidence": 0.5, "observation": {"observation_id": "test"}}
print(requests.post(f"{public_url}/resolve", json=payload, headers=headers, timeout=30).json())
